# Notebook 1: Data Preparation
**ANPR System – MIPA UGM Parking Lot**

This notebook:
1. Mounts Google Drive
2. Downloads Indonesian license plate datasets from Kaggle & Roboflow
3. Converts annotations to YOLO format
4. Merges datasets and removes duplicates
5. Splits into train/val/test (70/20/10)
6. Generates `data.yaml`
7. Visualizes samples and prints statistics

## 1. Setup & Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/ANPR_MIPA_UGM'

import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/datasets', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/processed', exist_ok=True)
print('Drive mounted and directories created.')

## 2. Install Dependencies

In [ ]:
!pip install -q kaggle roboflow ultralytics opencv-python-headless
import importlib
for pkg in ['kaggle', 'roboflow', 'ultralytics', 'cv2']:
    try:
        importlib.import_module(pkg)
        print(f'  {pkg} OK')
    except ImportError:
        print(f'  {pkg} MISSING')

## 3. Download from Kaggle

In [ ]:
# Upload kaggle.json first, or set credentials below
import os

KAGGLE_USERNAME = 'your_kaggle_username'  # @param {type:"string"}
KAGGLE_KEY = 'your_kaggle_api_key'        # @param {type:"string"}

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    import json
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

KAGGLE_DATASETS = [
    'malikahad/indonesian-license-plate-dataset',
    'juliagalang/car-plate-detection',
]

for ds in KAGGLE_DATASETS:
    name = ds.split('/')[-1]
    out_dir = f'{DRIVE_ROOT}/datasets/{name}'
    if not os.path.exists(out_dir):
        !kaggle datasets download -d {ds} -p {out_dir} --unzip
        print(f'Downloaded: {name}')
    else:
        print(f'Already exists: {name}')

## 4. Download from Roboflow

In [ ]:
from roboflow import Roboflow

RF_API_KEY = 'your_roboflow_api_key'  # @param {type:"string"}

rf = Roboflow(api_key=RF_API_KEY)

# Indonesian license plate detection project on Roboflow Universe
project = rf.workspace('roboflow-universe-projects').project('license-plate-recognition-rxg4e')
dataset = project.version(4).download('yolov8', location=f'{DRIVE_ROOT}/datasets/roboflow_lp')
print('Roboflow dataset downloaded.')

## 5. Convert to YOLO Format

In [ ]:
import os
import cv2
import json
import xml.etree.ElementTree as ET

def convert_voc_to_yolo(xml_path, class_map, img_w, img_h):
    """Convert Pascal VOC XML annotation to YOLO format."""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    lines = []
    for obj in root.findall('object'):
        class_name = obj.find('name').text
        if class_name not in class_map:
            continue
        class_id = class_map[class_name]
        bndbox = obj.find('bndbox')
        xmin = float(bndbox.find('xmin').text)
        ymin = float(bndbox.find('ymin').text)
        xmax = float(bndbox.find('xmax').text)
        ymax = float(bndbox.find('ymax').text)
        x_center = (xmin + xmax) / 2 / img_w
        y_center = (ymin + ymax) / 2 / img_h
        width = (xmax - xmin) / img_w
        height = (ymax - ymin) / img_h
        lines.append(f'{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}')
    return lines


def convert_coco_to_yolo(json_path, out_dir):
    """Convert COCO JSON annotations to YOLO format."""
    with open(json_path) as f:
        coco = json.load(f)
    
    img_map = {img['id']: img for img in coco['images']}
    cat_map = {cat['id']: i for i, cat in enumerate(coco['categories'])}
    annotations_by_image = {}
    for ann in coco['annotations']:
        img_id = ann['image_id']
        annotations_by_image.setdefault(img_id, []).append(ann)
    
    os.makedirs(out_dir, exist_ok=True)
    for img_id, img_info in img_map.items():
        w, h = img_info['width'], img_info['height']
        txt_name = os.path.splitext(img_info['file_name'])[0] + '.txt'
        lines = []
        for ann in annotations_by_image.get(img_id, []):
            cls = cat_map[ann['category_id']]
            x, y, bw, bh = ann['bbox']
            x_c = (x + bw / 2) / w
            y_c = (y + bh / 2) / h
            lines.append(f'{cls} {x_c:.6f} {y_c:.6f} {bw/w:.6f} {bh/h:.6f}')
        with open(os.path.join(out_dir, txt_name), 'w') as f:
            f.write('\n'.join(lines))

print('Conversion functions defined.')

## 6. Merge, Split, and Generate data.yaml

In [ ]:
import shutil
import random
from pathlib import Path

PROCESSED = Path(f'{DRIVE_ROOT}/processed')
for split in ['train', 'val', 'test']:
    (PROCESSED / 'images' / split).mkdir(parents=True, exist_ok=True)
    (PROCESSED / 'labels' / split).mkdir(parents=True, exist_ok=True)

# Collect all (image, label) pairs
all_pairs = []  # [(img_path, label_path), ...]

# --- Add your dataset scanning logic here ---
# Example: scan all directories for .jpg files with matching .txt labels
for ds_dir in Path(f'{DRIVE_ROOT}/datasets').rglob('*.jpg'):
    label_path = ds_dir.with_suffix('.txt')
    if label_path.exists():
        all_pairs.append((str(ds_dir), str(label_path)))

print(f'Total pairs found: {len(all_pairs)}')

# Shuffle and split
random.seed(42)
random.shuffle(all_pairs)
n = len(all_pairs)
n_train = int(n * 0.70)
n_val = int(n * 0.20)
splits = {'train': all_pairs[:n_train], 'val': all_pairs[n_train:n_train+n_val], 'test': all_pairs[n_train+n_val:]}

for split, pairs in splits.items():
    for i, (img, lbl) in enumerate(pairs):
        ext = Path(img).suffix
        dest_img = PROCESSED / 'images' / split / f'{split}_{i:05d}{ext}'
        dest_lbl = PROCESSED / 'labels' / split / f'{split}_{i:05d}.txt'
        shutil.copy2(img, dest_img)
        shutil.copy2(lbl, dest_lbl)
    print(f'{split}: {len(pairs)} images')

# Generate data.yaml
data_yaml = f"""path: {PROCESSED}
train: images/train
val: images/val
test: images/test
nc: 1
names: ['license_plate']
"""
yaml_path = PROCESSED / 'data.yaml'
with open(yaml_path, 'w') as f:
    f.write(data_yaml)
print(f'data.yaml saved to {yaml_path}')

## 7. Visualize Samples

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

def show_yolo_samples(image_dir, label_dir, n=12):
    images = sorted(Path(image_dir).glob('*.jpg'))[:n]
    fig, axes = plt.subplots(3, 4, figsize=(16, 10))
    for ax, img_path in zip(axes.flat, images):
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        lbl_path = Path(label_dir) / img_path.with_suffix('.txt').name
        if lbl_path.exists():
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        _, xc, yc, bw, bh = map(float, parts)
                        x1 = int((xc - bw/2) * w)
                        y1 = int((yc - bh/2) * h)
                        x2 = int((xc + bw/2) * w)
                        y2 = int((yc + bh/2) * h)
                        cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 2)
        ax.imshow(img)
        ax.set_title(img_path.name[:20], fontsize=8)
        ax.axis('off')
    plt.suptitle('Training Samples with Ground Truth Boxes', fontsize=14)
    plt.tight_layout()
    plt.show()

show_yolo_samples(
    f'{DRIVE_ROOT}/processed/images/train',
    f'{DRIVE_ROOT}/processed/labels/train'
)

## 8. Dataset Statistics

In [ ]:
from pathlib import Path

PROCESSED = Path(f'{DRIVE_ROOT}/processed')
stats = {}
for split in ['train', 'val', 'test']:
    imgs = list((PROCESSED / 'images' / split).glob('*.jpg'))
    lbls = list((PROCESSED / 'labels' / split).glob('*.txt'))
    total_boxes = sum(sum(1 for l in open(lb) if l.strip()) for lb in lbls)
    stats[split] = {'images': len(imgs), 'labels': len(lbls), 'boxes': total_boxes}

print('Dataset Statistics')
print('-' * 40)
print(f'{"Split":<10} {"Images":<10} {"Labels":<10} {"Boxes":<10}')
for split, s in stats.items():
    print(f'{split:<10} {s["images"]:<10} {s["labels"]:<10} {s["boxes"]:<10}')
total_imgs = sum(s['images'] for s in stats.values())
total_boxes = sum(s['boxes'] for s in stats.values())
print('-' * 40)
print(f'Total: {total_imgs} images, {total_boxes} bounding boxes')